# L2b: Errors, Tests, and Debugging a Numerical Program

A program can run without throwing an exception and still be wrong. A known physical case and a small test will expose a unit-conversion defect.

> **Learning Objectives**
> 1. Distinguish syntax, runtime, and incorrect-result failures.
> 2. Turn a unit contract into a reference test.
> 3. Use an error message and a failing expectation to localize a defect.
> 4. Compare equivalent Julia and Python interfaces.

---

## Setup, Data, and Prerequisites

Run the local setup cell first. It activates the single pinned course environment, loads every package used by this meeting, and includes any meeting source code.

In [ ]:
include(joinpath(@__DIR__, "Include.jl"))

## Specification and reference case

Residence time is $\tau=V/Q$. Our interface accepts volume in L and flow in mL/min and returns min. Therefore a 2 L vessel at 250 mL/min has a residence time of 8 min.

In [ ]:
function buggy_residence_time_minutes(volume_L, flow_mL_min)
    return volume_L / flow_mL_min
end

buggy_result = buggy_residence_time_minutes(2.0, 250.0)

The function executed, but the known-case expectation fails. Compare the magnitude and inspect the units before changing code.

In [ ]:
expected_minutes = 8.0
diagnostic = (
    result = buggy_result,
    expected = expected_minutes,
    passes = isapprox(buggy_result, expected_minutes),
    missing_scale_factor = expected_minutes / buggy_result,
)

## Repair the units and defend the boundary

The volume must be converted from L to mL before division. The shared implementation also rejects zero, negative, and non-finite inputs.

In [ ]:
correct_result = residence_time_minutes(2.0, 250.0)
(correct_result = correct_result, agrees = correct_result == expected_minutes)

## Exceptions are part of the interface

Invalid inputs should fail near the interface with a specific message. Catch an error only when the program can make a meaningful decision about it.

In [ ]:
caught_message = try
    residence_time_minutes(2.0, 0.0)
    "no error"
catch error
    sprint(showerror, error)
end

## Same contract in Python

The companion Python implementation uses `TypeError` for the wrong kind of input and `ValueError` for an invalid numerical value. The language syntax differs; the contract and reference test do not.

In [ ]:
python_source = joinpath(CHEME5800_L2B_ROOT, "src", "residence_time.py")
python_preview = join(first(split(read(python_source, String), '\n'), 18), "\n")
python_preview

Run the Python comparison from the bundle root:

```bash
python -m unittest discover -s weeks/week-02/L2b/src -p 'test_*.py'
```

## Turn the diagnosis into regression tests

In [ ]:

@testset "errors, tests, and debugging" begin
    @test buggy_result != expected_minutes
    @test diagnostic.missing_scale_factor == 1000.0
    @test correct_result == expected_minutes
    @test residence_time_minutes(0.5, 100) == 5.0
    @test_throws ArgumentError residence_time_minutes(0, 100)
    @test_throws ArgumentError residence_time_minutes(1, Inf)
    @test occursin("flow_mL_min", caught_message)
    @test isfile(python_source)
end

## Summary

> **Key Takeaways**
> 1. Running without an exception is not evidence of correctness.
> 2. Reference cases and dimensional analysis are debugging tools.
> 3. Preserve the same behavioral contract when comparing languages.

---